In [ ]:
import os
import re
from glob import glob

import joblib
import numpy as np
import rasterio
from rasterio.errors import NotGeoreferencedWarning

import warnings
warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)

In [ ]:
# ==========================
# CONFIGURACIÓN
# ==========================
DATA_DIR = r"C:\SPA_SEI\presentaciones\data_cut"
PATTERN  = re.compile(r"^(\d{8})_multi\.tif$", re.IGNORECASE)

SCALER_PATH = os.path.join(DATA_DIR, "scaler.joblib")
MODEL_PATH  = os.path.join(DATA_DIR, "kmeans.joblib")

OUTPUT_SUFFIX = "_class.tif"   # etiquetas 1-5
OVERWRITE = True

In [ ]:
# ==========================
# UTILIDADES
# ==========================
def list_multis(folder):
    files = sorted(glob(os.path.join(folder, "*_multi.tif")))
    items = []
    for f in files:
        name = os.path.basename(f)
        m = PATTERN.match(name)
        if m:
            date = m.group(1)
            items.append((date, f))
    return items

def classify_one(path, scaler, kmeans):
    """Clasifica una imagen multibanda (VV,VH,PR), retornando (labels_uint8, meta_salida)"""
    with rasterio.open(path) as src:
        if src.count < 3:
            raise ValueError(f"{os.path.basename(path)} no tiene 3 bandas (VV,VH,PR).")
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)
        pr = src.read(3).astype(np.float32)

        nodata = src.nodata  # de VV, por diseño
        if nodata is not None:
            mask = (vv != nodata)
        else:
            mask = np.isfinite(vv) & np.isfinite(vh) & np.isfinite(pr)

        mask &= np.isfinite(vv) & np.isfinite(vh) & np.isfinite(pr)

        H, W = vv.shape
        X = np.stack([vv, vh, pr], axis=2).reshape(-1, 3)

        valid_idx = mask.reshape(-1)
        X_valid = X[valid_idx]

        # Escalar y predecir
        X_scaled = scaler.transform(X_valid)
        labels = kmeans.predict(X_scaled)  # 0..K-1

        # Convertir a 1-5
        labels_1k = (labels + 1).astype(np.uint8)

        # Reconstruir raster de clases
        out = np.zeros(H * W, dtype=np.uint8)
        out[:] = 0  # 0 = background si hubiera celdas inválidas (no debería si no hay nodata)
        out[valid_idx] = labels_1k
        out = out.reshape(H, W)

        # Meta de salida: igual al multibanda, 1 banda
        meta = src.meta.copy()
        meta.update({
            "count": 1,
            "dtype": "uint8",
            "nodata": 0  # 0 como fondo (si VV tenía nodata, quedará 0 en salida)
        })
        for k in ["compress", "tiled", "predictor", "zlevel"]:
            meta.pop(k, None)

        return out, meta

In [ ]:
# ==========================
# MAIN
# ==========================
def main():
    if not (os.path.exists(SCALER_PATH) and os.path.exists(MODEL_PATH)):
        raise SystemExit("No se encontraron scaler.joblib y/o kmeans.joblib. Ejecuta primero train_kmeans.py")

    scaler = joblib.load(SCALER_PATH)
    kmeans = joblib.load(MODEL_PATH)

    items = list_multis(DATA_DIR)
    if not items:
        raise SystemExit("No se encontraron *_multi.tif en la carpeta de entrada.")

    for date, in_path in items:
        out_path = os.path.join(DATA_DIR, f"{date}{OUTPUT_SUFFIX}")
        if (not OVERWRITE) and os.path.exists(out_path):
            print(f"[SALTA] Ya existe {os.path.basename(out_path)} y OVERWRITE=False")
            continue

        labels_raster, meta = classify_one(in_path, scaler, kmeans)

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(labels_raster, 1)

        print(f"[OK] Clasificado: {os.path.basename(out_path)}")

if __name__ == "__main__":
    main()